In [20]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import textwrap
import re

# Configure TextWrapper
wrapper = textwrap.TextWrapper(width=70)

# Set random seed for reproducibility
tf.keras.utils.set_random_seed(10)

# Define utility functions
def get_train_test_data(data_dir):
    """
    Loads train and test data from the specified directory.
    """
    train_data = pd.read_json(f"{data_dir}/train.json")
    train_data.drop(['id'], axis=1, inplace=True)

    test_data = pd.read_json(f"{data_dir}/test.json")
    test_data.drop(['id'], axis=1, inplace=True)

    return train_data, test_data

def preprocess(input_data):
    """
    Preprocesses the summary and dialogue text in the dataset.
    """
    def preprocess_util(text):
        # Convert all text to lowercase
        lowercase = text.lower()
        # Remove newlines and extra spaces
        removed_newlines = re.sub("\n|\r|\t", " ", lowercase)
        removed_double_spaces = ' '.join(removed_newlines.split())
        # Add start of sentence and end of sentence tokens
        processed_text = '[SOS] ' + removed_double_spaces + ' [EOS]'
        return processed_text

    input_data['summary'] = input_data['summary'].apply(preprocess_util)
    input_data['dialogue'] = input_data['dialogue'].apply(preprocess_util)
    return input_data

# Specify the directory containing the dataset
data_dir = r'C:\Users\admin\Desktop\NLP\corpus\corpus'

# Load and preprocess data
train_data, test_data = get_train_test_data(data_dir)
train_data = preprocess(train_data)
test_data = preprocess(test_data)

# Extract the 'dialogue' and 'summary' columns after preprocessing
document = train_data['dialogue']
summary = train_data['summary']

document_test = test_data['dialogue']
summary_test = test_data['summary']

# Tokenizer Configuration
filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
oov_token = '[UNK]'

# Initialize and fit tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters=filters, oov_token=oov_token, lower=False)
document_and_summary = pd.concat([document, summary], ignore_index=True)
tokenizer.fit_on_texts(document_and_summary)

# Prepare Inputs and Targets
inputs = tokenizer.texts_to_sequences(document)
targets = tokenizer.texts_to_sequences(summary)

# Vocabulary Size
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding token

# Example Output
print("\nTokenizer Vocabulary Size:", vocab_size)
print("\nExample Input Sequence:", inputs[0])
print("\nExample Target Sequence:", targets[0])


Tokenizer Vocabulary Size: 34237

Example Input Sequence: [7, 454, 2, 3500, 1611, 30, 5, 81, 50, 617, 66, 454, 63, 220, 5, 98, 8]

Example Target Sequence: [7, 454, 3500, 1611, 9, 15, 220, 617, 50, 98, 8]


In [28]:
# Limit the size of the input and output data for processing in this environment.
encoder_maxlen = 150
decoder_maxlen = 50

# Pad the sequences.
inputs = tf.keras.preprocessing.sequence.pad_sequences(
    inputs, maxlen=encoder_maxlen, padding='post', truncating='post'
)
targets = tf.keras.preprocessing.sequence.pad_sequences(
    targets, maxlen=decoder_maxlen, padding='post', truncating='post'
)

# Convert inputs and targets to tensors.
inputs = tf.cast(inputs, dtype=tf.int32)
targets = tf.cast(targets, dtype=tf.int32)

# Create the final training dataset.
BUFFER_SIZE = 10000
BATCH_SIZE = 64
dataset = tf.data.Dataset.from_tensor_slices((inputs, targets)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

In [29]:
# Pad the sequences.
inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs, maxlen=encoder_maxlen, padding='post', truncating='post')
targets = tf.keras.preprocessing.sequence.pad_sequences(targets, maxlen=decoder_maxlen, padding='post', truncating='post')

In [30]:
def positional_encoding(positions, d_model):

    # Create a matrix of positions and dimensions
    position = np.arange(positions)[:, np.newaxis]
    k = np.arange(d_model)[np.newaxis, :]
    
    # Compute the angle rates
    angle_rates = 1 / np.power(10000, (2 * (k // 2)) / np.float32(d_model))
    angle_rads = position * angle_rates

    # Apply sin to even indices (2i) in the array
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # Apply cos to odd indices (2i+1) in the array
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    # Add an extra dimension to match the required shape
    pos_encoding = angle_rads[np.newaxis, ...]

    return tf.cast(pos_encoding, dtype=tf.float32)

In [31]:
def create_padding_mask(decoder_token_ids):
   
    # Create a binary mask where padding tokens (ID=8) are marked as 1
    seq = tf.cast(tf.math.equal(decoder_token_ids, 8), tf.float32)
    
    # Add an extra dimension to the mask to allow broadcasting during attention calculations
    return seq[:, tf.newaxis, :]


In [32]:
def create_look_ahead_mask(sequence_length):

    # Create a lower triangular matrix filled with ones
    mask = tf.linalg.band_part(tf.ones((sequence_length, sequence_length)), -1, 0)
    
    return mask


In [34]:
def scaled_dot_product_attention(q, k, v, mask):

    # Multiply query (q) and key (k^T)
    matmul_qk = tf.matmul(q, k, transpose_b=True)  # Shape: (..., seq_len_q, seq_len_k)
    
    # Scale matmul_qk by the square root of the key's depth (dk)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)  # Depth of the key
    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
    
    # Add the mask to the scaled tensor (if provided)
    if mask is not None:
        # Masking: Use a very large negative value to effectively zero out unwanted positions
        scaled_attention_logits += (mask * -1e9)
    
    # Apply softmax to get normalized attention weights
    # Softmax is applied on the last axis (seq_len_k) so scores add up to 1
    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)  # Shape: (..., seq_len_q, seq_len_k)
    
    # Multiply attention weights by value (v)
    output = tf.matmul(attention_weights, v)  # Shape: (..., seq_len_q, depth_v)
    
    return output, attention_weights


In [35]:
def FullyConnected(embedding_dim, fully_connected_dim):

    return tf.keras.Sequential([
        tf.keras.layers.Dense(fully_connected_dim, activation='relu'),  # First dense layer
        tf.keras.layers.Dense(embedding_dim)  # Second dense layer, no activation
    ])


In [39]:
class EncoderLayer(tf.keras.layers.Layer):
 
    def __init__(self, embedding_dim, num_heads, fully_connected_dim, dropout_rate=0.1, layernorm_eps=1e-6):
     
        super(EncoderLayer, self).__init__()

        # Multi-Head Attention layer
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim,
            dropout=dropout_rate
        )

        # Feed-Forward Network (FFN)
        self.ffn = FullyConnected(embedding_dim=embedding_dim, fully_connected_dim=fully_connected_dim)

        # Layer Normalization layers
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)

        # Dropout layer for the FFN output
        self.dropout_ffn = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training, mask):
        
        # Calculate self-attention using mha
        # Self attention (batch_size, input_seq_len, embedding_dim)
        self_mha_output = self.mha(x, x, x, attention_mask=mask)

        # Skip connection: add the input and the self-attention output
        # Apply layer normalization on the sum of the input and the attention output
        skip_x_attention = self.layernorm1(x + self_mha_output)  # (batch_size, seq_len, embedding_dim)

        # Pass the output of the multi-head attention layer through the feed-forward network (FFN)
        ffn_output = self.ffn(skip_x_attention)  # (batch_size, seq_len, embedding_dim)

        # Apply dropout to FFN output during training
        ffn_output = self.dropout_ffn(ffn_output, training=training)

        # Apply layer normalization on the sum of the output from multi-head attention (skip connection)
        # and the FFN output to get the final output of the encoder layer
        encoder_layer_out = self.layernorm2(skip_x_attention + ffn_output)  # (batch_size, seq_len, embedding_dim)

        return encoder_layer_out


In [ ]:
class Encoder(tf.keras.layers.Layer):


    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, maximum_position_encoding, dropout_rate=0.1, layernorm_eps=1e-6):
 
        super(Encoder, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        
        # Embedding layer for input sequence
        self.embedding = tf.keras.layers.Embedding(input_vocab_size, self.embedding_dim)
        
        # Positional encoding
        self.pos_encoding = positional_encoding(maximum_position_encoding, self.embedding_dim)
        
        # Encoder layers
        self.enc_layers = [
            EncoderLayer(embedding_dim=self.embedding_dim, num_heads=num_heads, fully_connected_dim=fully_connected_dim, dropout_rate=dropout_rate, layernorm_eps=layernorm_eps)
            for _ in range(self.num_layers)
        ]
        
        # Dropout layer for regularization
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training, mask):
  
        # Add embeddings to the input sequence and apply positional encoding
        x = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        x += self.pos_encoding  # Add positional encoding
        x = self.dropout(x, training=training)  # Apply dropout to embeddings
        
        # Pass through the stack of encoder layers
        for enc_layer in self.enc_layers:
            x = enc_layer(x, training, mask)  # (batch_size, seq_len, embedding_dim)
        
        return x
